# US Accidents - Advanced Data Cleaning (Imputation)
ใน Notebook นี้เราจะดำเนินการจัดการกับค่าที่หายไป (Missing Values) ที่เหลืออยู่ โดยใช้เทคนิค **Hierarchical Imputation** เพื่อเติมค่าตามลำดับความสัมพันธ์ของพื้นที่และเวลา

## 1. Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

sys.path.append('../src')

import util as ut

In [2]:
df_train = pd.read_csv("../data/processed/01.2/train_data.csv")
df_test = pd.read_csv("../data/processed/01.2/test_data.csv")

In [3]:
df_train["Start_Time"] = pd.to_datetime(df_train["Start_Time"], errors="coerce")
df_test["Start_Time"] = pd.to_datetime(df_test["Start_Time"], errors="coerce")

## 2. Initial Missing Value Analysis

In [4]:
missing_data = df_train.isnull().sum()[df_train.isnull().sum() > 0].sort_values(ascending=False)
print("--- Number of Null ---")
print(missing_data)

In [5]:
missing_percent = (df_train.isnull().sum() / len(df_train)) * 100
missing_df = missing_percent[missing_percent > 0].sort_values(ascending=False)
print("--- Percent of Null ---")
print(missing_df)

## 3. Weather Condition Preprocessing & Grouping

In [6]:
df_train['Weather_Condition'] = df_train['Weather_Condition'].str.lower()
df_test['Weather_Condition'] = df_test['Weather_Condition'].str.lower()

In [7]:
df_train['Start_Date'] = df_train['Start_Time'].dt.date
df_train['Hour'] = df_train['Start_Time'].dt.hour
df_train['Month'] = df_train['Start_Time'].dt.month

df_test['Start_Date'] = df_test['Start_Time'].dt.date
df_test['Hour'] = df_test['Start_Time'].dt.hour
df_test['Month'] = df_test['Start_Time'].dt.month

In [8]:
weather_first_train = df_train.groupby(['City', 'Start_Date', 'Hour', 'Street'])['Weather_Condition'].transform('first')
df_train['Weather_Condition'] = df_train['Weather_Condition'].fillna(weather_first_train)

weather_first_test = df_test.groupby(['City', 'Start_Date', 'Hour', 'Street'])['Weather_Condition'].transform('first')
df_test['Weather_Condition'] = df_test['Weather_Condition'].fillna(weather_first_test)

In [9]:
df_train = ut.apply_weather_grouping(df_train)
df_test = ut.apply_weather_grouping(df_test)
df_train = df_train.drop(columns=['Weather_Condition'])
df_test = df_test.drop(columns=['Weather_Condition'])

## 4. Hierarchical Imputation for Numerical & Categorical Features

In [10]:
# Hierarchical Imputation
from pandas.api.types import is_numeric_dtype, is_string_dtype, is_object_dtype

hierarchies = [
    ['City', 'Start_Date', 'Hour'],  
    ['City', 'Start_Date'],        
    ['City', 'Month'],           
    ['Weather_Group']            
]

num_weather_cols = [
    'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 
    'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)', "Wind_Direction"
]

print("Hierarchical Imputation for df_train...")
for col in num_weather_cols:
    for group_cols in hierarchies:
        if df_train[col].isnull().sum() == 0: break
        if is_numeric_dtype(df_train[col]):
            df_train[col] = df_train[col].fillna(df_train.groupby(group_cols)[col].transform('median'))
        elif is_string_dtype(df_train[col]) or is_object_dtype(df_train[col]):
            df_train[col] = df_train[col].fillna(df_train.groupby(group_cols)[col].transform('first'))

print("Hierarchical Imputation for df_test (Using stats from df_train to prevent leak)...")
for col in num_weather_cols:
    # 1. First fill with df_train group stats
    for group_cols in hierarchies:
        if df_test[col].isnull().sum() == 0: break
        
        # Calculate stats from train
        if is_numeric_dtype(df_train[col]):
            stats_map = df_train.groupby(group_cols)[col].median()
        else:
            stats_map = df_train.groupby(group_cols)[col].first()
            
        # Map to test
        df_test = df_test.merge(stats_map.reset_index().rename(columns={col: col+'_impute'}), on=group_cols, how='left')
        df_test[col] = df_test[col].fillna(df_test[col+'_impute'])
        df_test = df_test.drop(columns=[col+'_impute'])

    # 2. Final fallback: Global train median/mode
    if df_test[col].isnull().sum() > 0:
        if is_numeric_dtype(df_train[col]):
            df_test[col] = df_test[col].fillna(df_train[col].median())
        else:
            df_test[col] = df_test[col].fillna(df_train[col].mode()[0])

## 5. Verification and Data Export

In [11]:
missing_percent = (df_train.isnull().sum() / len(df_train)) * 100
missing_df = missing_percent[missing_percent > 0].sort_values(ascending=False)
print("--- Percent of Null ---")
print(missing_df)

In [12]:
save_dir = "../data/processed/01.3"
import os
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
df_train.to_csv(f"{save_dir}/train_advance_clean.csv", index=False)
df_test.to_csv(f"{save_dir}/test_advance_clean.csv", index=False)
print(f"Saved processed data to {save_dir}")